# 01 — Process
End-to-end processing pipeline: merge → calibrate CTD → QC → regrid → CF align → save.

In [ ]:
import sys, yaml
import numpy as np
import xarray as xr
sys.path.insert(0, '..')
sys.path.append("../adcp")
from adcp import merge, physics, ctd as ctd_mod, gap_fill, qc, regrid, cf
from ooi_data_explorations.qartod import gross_range, climatology

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
config = yaml.safe_load(open('../config/GI01SUMO-RII11-02-ADCPSN010.yaml'))
refdes  = config['refdes']
data_dir    = '../data/'
results_dir = '../results/'

## 1. Merge ADCP streams

In [ ]:
tdata = xr.open_dataset(f'{data_dir}{refdes}.telemetered.raw.nc').load()
tdata = tdata.sortby('time')
hdata = xr.open_dataset(f'{data_dir}{refdes}.recovered_host.raw.nc').load()
hdata = hdata.sortby('time')
idata = xr.open_dataset(f'{data_dir}{refdes}.recovered_inst.raw.nc').load()
idata = idata.sortby('time')

In [ ]:
adcp = merge.merge_adcp_streams(
    tdata, hdata, idata,
    deployments_to_drop=config.get('deployments_to_drop'),
)
adcp

## 2. Calibrate CTD against bottle data

In [ ]:
ctd_refdes = config['ctd_refdes']
ctd = xr.open_dataset(f'{data_dir}{ctd_refdes}.merged.nc')

cal_ctd, corrections, stats = ctd_mod.calibrate_ctd(
    ctd,
    bottle_path=f'{data_dir}cleaned_bottle_data.csv',
    output_dir=results_dir,
    mooring_pressure=config['qc']['mooring_pressure'],
    mooring_pressure_window=config['qc']['mooring_pressure_window'],
)

## 3. Gap-fill calibrated CTD

In [ ]:
ctd_filled, fill_results = gap_fill.fill_ctd_gaps(cal_ctd)
ctd_filled.to_netcdf(
    f'{data_dir}{ctd_refdes}.calibrated.filled.nc',
    format='netcdf4', engine='h5netcdf',
)
ctd_filled

## 4. Merge CTD into ADCP, recalculate sound speed

In [ ]:
adcp = ctd_mod.merge_ctd_into_adcp(adcp, ctd_filled)
adcp

## 5. Sensor engineering QC (TRDI + roll/pitch/sidelobe)

In [ ]:
combined_qc, combined_attrs = qc.compute_sensor_engineering_qc(adcp, config)
adcp['qc_flags'] = (['time', 'bin'], combined_qc)
adcp['qc_flags'].attrs = combined_attrs
combined_attrs

## 6. Regrid to common depth grid

In [ ]:
depth_grid = regrid.build_depth_grid(config)
print(f'Depth grid: {depth_grid[0]:.0f}–{depth_grid[-1]:.0f} m, step {config["regrid"]["depth_step"]} m, {len(depth_grid)} levels')

adcp_g = regrid.regrid_adcp(adcp, depth_grid)
adcp_g = merge.drop_processing_vars(adcp_g)
adcp_g

## 7. QARTOD — scalar CTD variables

In [ ]:
qc_cfg = config['qc']

for var, fail_key in [
    ('sea_water_temperature',        'temperature_fail_range'),
    ('sea_water_practical_salinity', 'salinity_fail_range'),
]:
    fail_range = tuple(qc_cfg.get(fail_key, [-999, 999]))
    long_name  = adcp_g[var].attrs.get('long_name', var)

    gr = gross_range.GrossRange(*fail_range)
    gr.fit(adcp_g, var, sigma=3, check_normality=True)

    clm = climatology.Climatology()
    clm.fit(adcp_g[var])

    range_flags = qc.qartod_range_test(
        adcp_g[var].values, (gr.fail_min, gr.fail_max), (gr.suspect_min, gr.suspect_max),
    )
    clim_flags = qc.qartod_climatology_test(
        adcp_g[var].values, adcp_g['time'].values,
        np.array(clm.monthly_mu), np.array(clm.monthly_std),
        fail_range, qc_cfg.get('climatology_suspect_std', 2),
    )

    test_names = ['gross_range_test', 'climatology_test']
    executed, exec_attrs = qc.zip_flags([range_flags, clim_flags], test_names, var, long_name)
    summary,  sum_attrs  = qc.combine_flags([range_flags, clim_flags], test_names, var, long_name)

    adcp_g[f'{var}_qartod_executed'] = (['time'], executed)
    adcp_g[f'{var}_qartod_executed'].attrs = exec_attrs
    adcp_g[f'{var}_qartod_results']  = (['time'], summary)
    adcp_g[f'{var}_qartod_results'].attrs  = sum_attrs
    print(f'{var}: done')

## 8. QARTOD — 2-D velocity variables

In [ ]:
fail_range = tuple(qc_cfg.get('velocity_fail_range', [-5, 5]))
sus_std    = qc_cfg.get('velocity_suspect_std', 3)
woa_depths = qc.WOA_STANDARD_DEPTHS[qc.WOA_STANDARD_DEPTHS <= adcp_g['bin_depths'].max().item()]

for var in ('eastward_seawater_velocity', 'northward_seawater_velocity'):
    executed, summary, exec_attrs, sum_attrs = qc.run_velocity_qartod(
        adcp_g, var, fail_range, woa_depths, sus_std,
    )
    adcp_g[f'{var}_qartod_executed'] = (['time', 'bin_depths'], executed)
    adcp_g[f'{var}_qartod_executed'].attrs = exec_attrs
    adcp_g[f'{var}_qartod_results']  = (['time', 'bin_depths'], summary)
    adcp_g[f'{var}_qartod_results'].attrs  = sum_attrs
    print(f'{var}: done')

## 9. CF-1.11 alignment

In [ ]:
adcp_g = cf.fix_parameter_names(adcp_g)
adcp_g = cf.fix_parameter_attrs(adcp_g)
adcp_g = cf.combine_adcp_beam_params(adcp_g)
adcp_g = cf.fix_coordinates(adcp_g)
adcp_g = cf.fix_cf_compliance(adcp_g)

adcp_final = adcp_g.copy()
adcp_final.attrs = cf.clean_global_attrs(adcp_g)
adcp_final = cf.add_station_id(adcp_final)
adcp_final = cf.sanitize_attrs(adcp_final)
adcp_final

## 10. Save

In [ ]:
outpath = f'{data_dir}{refdes}.nc'
adcp_final.to_netcdf(outpath, format='netcdf4', engine='h5netcdf')
print(f'Saved → {outpath}')